# CUDA Performance Analysis: Vector Addition & Matrix Multiplication

This notebook benchmarks three implementations:
- **Sequential CPU** — single-threaded baseline
- **Global Memory GPU** — naive parallel (reads directly from VRAM each time)
- **Shared Memory GPU** — tiled parallel (caches tiles in fast on-chip memory)

> GPU timing uses **CUDA Events** (not wall-clock) for kernel-only precision, averaged over 5 runs with a warm-up pass to eliminate cold-start noise.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

---
## Part 1 — Matrix Multiplication

In [ ]:
mat = pd.read_csv('vector_matrix_result.txt')
print('Matrix Benchmark Results:')
print(mat.to_string(index=False))

### 1.1 Execution Time: Sequential CPU vs Parallel GPU

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(mat['SIZE'], mat['CPU_SEQ_TIME'],   marker='o', linewidth=2, label='Sequential CPU')
ax.plot(mat['SIZE'], mat['GPU_GLOBAL_TIME'], marker='s', linewidth=2, label='Parallel GPU (Global Memory)')
ax.plot(mat['SIZE'], mat['GPU_SHARED_TIME'], marker='^', linewidth=2, label='Parallel GPU (Shared Memory)')

ax.set_title('Matrix Multiplication Execution Time\n(Sequential vs Parallel)', fontsize=13)
ax.set_xlabel('Matrix Size N (N × N)', fontsize=11)
ax.set_ylabel('Time (ms, log scale)', fontsize=11)
ax.set_yscale('log')
ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax.legend()
ax.grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 1.2 Speedup over Sequential CPU

**Speedup** = $T_{CPU\_Sequential}$ / $T_{GPU\_Parallel}$

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(mat['SIZE'], mat['SPEEDUP_GLOBAL'], marker='s', linewidth=2, color='steelblue',  label='Global Memory Speedup')
ax.plot(mat['SIZE'], mat['SPEEDUP_SHARED'], marker='^', linewidth=2, color='darkorange', label='Shared Memory Speedup')

ax.set_title('GPU Speedup over Sequential CPU — Matrix Multiplication', fontsize=13)
ax.set_xlabel('Matrix Size N (N × N)', fontsize=11)
ax.set_ylabel('Speedup (×)', fontsize=11)
ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 1.3 Shared Memory Advantage over Global Memory

**Shared Advantage** = $T_{Global\_GPU}$ / $T_{Shared\_GPU}$

A value > 1 means Shared Memory was faster. Because CUDA Events are used for precise timing and a **warm-up run is performed**, this metric now correctly reflects the benefit of caching — not kernel launch latency noise. Notice the advantage grows consistently with N as cache reuse becomes more impactful.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(mat['SIZE'].astype(str), mat['SHARED_ADVANTAGE'], color=sns.color_palette('viridis', len(mat)), edgecolor='black', width=0.5)
ax.axhline(1.0, color='red', linestyle='--', linewidth=1.5, label='No advantage (=1.0)')

for bar, val in zip(bars, mat['SHARED_ADVANTAGE']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.2f}×', ha='center', va='bottom', fontsize=11)

ax.set_title('Shared Memory Advantage over Global Memory\n(Global Time / Shared Time)', fontsize=13)
ax.set_xlabel('Matrix Size N (N × N)', fontsize=11)
ax.set_ylabel('Advantage Factor (×)', fontsize=11)
ax.legend()
ax.set_ylim(0, mat['SHARED_ADVANTAGE'].max() * 1.2)
plt.tight_layout()
plt.show()

print(f'Average Shared Memory advantage: {mat["SHARED_ADVANTAGE"].mean():.2f}×')

---
## Part 2 — Vector Addition: Sequential vs Parallel

In [ ]:
vec = pd.read_csv('vector_result.txt')
print('Vector Add Benchmark Results:')
print(vec.to_string(index=False))

### 2.1 Execution Time: Sequential CPU vs Parallel GPU

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(vec['SIZE'], vec['CPU_SEQ_TIME'],     marker='o', linewidth=2, label='Sequential CPU')
ax.plot(vec['SIZE'], vec['GPU_PARALLEL_TIME'], marker='s', linewidth=2, label='Parallel GPU')

ax.set_title('Vector Addition Execution Time\n(Sequential vs Parallel)', fontsize=13)
ax.set_xlabel('Vector Size (elements)', fontsize=11)
ax.set_ylabel('Time (ms)', fontsize=11)
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 2.2 Speedup over Sequential CPU

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(vec['SIZE'], vec['SPEEDUP'], marker='o', linewidth=2, color='green', label='GPU Speedup')
ax.fill_between(vec['SIZE'], 1, vec['SPEEDUP'], alpha=0.15, color='green')
ax.axhline(1.0, color='gray', linestyle='--', linewidth=1.5, label='Baseline (Sequential = 1×)')

ax.set_title('Vector Addition: GPU Speedup over Sequential CPU', fontsize=13)
ax.set_xlabel('Vector Size (elements)', fontsize=11)
ax.set_ylabel('Speedup (×)', fontsize=11)
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f'Average GPU speedup for Vector Addition: {vec["SPEEDUP"].mean():.1f}×')